# Model training — ConvNeXt-Small 

In [ ]:
import os, random, time, copy
import numpy as np
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader
from torchvision import models, datasets, transforms
import torchvision.transforms.functional as TF

In [ ]:
torch.__version__

'2.12.1+cu130'

In [ ]:
_cwd = Path(os.getcwd())
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
print("Project root:", PROJECT_ROOT)

data_path       = PROJECT_ROOT / "data" / "dataset_split_rafdb"
model_save_path = PROJECT_ROOT / "convnext_small_rafdb.pth"

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
seed = 64
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True  # reproducible runs at a small speed cost
torch.backends.cudnn.benchmark = False

### Data loading

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),  # mild crop: faces are already tightly aligned
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    normalize,
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.20)),  # simulates occlusion (hair, hand, glasses)
])

val_tfms = transforms.Compose([
    transforms.Resize(224, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    normalize,
])

image_datasets = {
    "train": datasets.ImageFolder(data_path / "train", transform=train_tfms),
    "val":   datasets.ImageFolder(data_path / "val",   transform=val_tfms),
    "test":  datasets.ImageFolder(data_path / "test",  transform=val_tfms),
}

num_classes   = len(image_datasets["train"].classes)
train_targets = [y for _, y in image_datasets["train"].samples]
counts        = Counter(train_targets)

# Sqrt (not full inverse-frequency) resampling: softer correction, avoids over-sampling rare classes into overfitting
class_sample_weights = 1.0 / torch.tensor([counts[i] for i in range(num_classes)], dtype=torch.float).sqrt()
sample_weights = class_sample_weights[train_targets]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

dataloaders = {
    "train": DataLoader(image_datasets["train"], batch_size=32, sampler=sampler,  num_workers=4, pin_memory=True),
    "val":   DataLoader(image_datasets["val"],   batch_size=32, shuffle=False, num_workers=4, pin_memory=True),
    "test":  DataLoader(image_datasets["test"],  batch_size=32, shuffle=False, num_workers=4, pin_memory=True),
}

print("Classes:", image_datasets["train"].classes)
print("Train class counts:", dict(counts))

In [ ]:
class FocalLoss(nn.Module):
    # Down-weights easy/majority-class examples, rare classes contribute more to the gradient.
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        if weight is not None and not torch.is_tensor(weight):
            weight = torch.tensor(weight, dtype=torch.float32)
        self.register_buffer("weight", weight)
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight,
                             reduction="none", label_smoothing=self.label_smoothing)
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

In [ ]:
# On top of the sampler: stronger, explicit per-class weighting in the loss itself.
class_weights = torch.tensor([1.0 / counts[i] for i in range(num_classes)], dtype=torch.float)
class_weights = class_weights / class_weights.sum() * num_classes
class_weights = class_weights.to(device)
print("Class weights:", {
    image_datasets["train"].classes[i]: f"{class_weights[i].item():.3f}"
    for i in range(num_classes)
})

criterion = FocalLoss(weight=class_weights, gamma=2.0, label_smoothing=0.1).to(device)

### Training

In [ ]:
def _cutmix(inputs, labels, alpha, device):
    # Pastes a rectangular patch from one sample onto another; label mixed by the patch's area share.
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(inputs.size(0), device=device)
    H, W = inputs.size(2), inputs.size(3)
    cut_h = int(H * np.sqrt(1 - lam))
    cut_w = int(W * np.sqrt(1 - lam))
    cy, cx = np.random.randint(H), np.random.randint(W)
    y1, y2 = max(0, cy - cut_h // 2), min(H, cy + cut_h // 2)
    x1, x2 = max(0, cx - cut_w // 2), min(W, cx + cut_w // 2)
    mixed = inputs.clone()
    mixed[:, :, y1:y2, x1:x2] = inputs[idx, :, y1:y2, x1:x2]
    lam = 1 - (y2 - y1) * (x2 - x1) / (H * W)
    return mixed, labels[idx], lam


def train_model(model, dataloaders, device,
                criterion, optimizer, num_epochs=25, scheduler=None,
                early_stopping=True, patience=15, min_delta=1e-4,
                mixup_alpha=0.0, cutmix_alpha=0.0):

    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    epochs_no_improve = 0

    dataset_size = {k: len(dataloaders[k].dataset) for k in ["train", "val"]}
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    use_amp = (device.type == "cuda")
    scaler  = GradScaler("cuda", enabled=use_amp)  # mixed-precision: faster/lower-memory GPU training
    is_onecycle = isinstance(scheduler, torch.optim.lr_scheduler.OneCycleLR)

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        print("-" * 10)

        for phase in ["train", "val"]:
            is_train = (phase == "train")
            model.train() if is_train else model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                lam, labels_b = 1.0, labels
                if is_train and (mixup_alpha > 0 or cutmix_alpha > 0):
                    # randomly alternate between MixUp and CutMix each batch
                    use_cutmix = cutmix_alpha > 0 and (mixup_alpha == 0 or np.random.rand() < 0.5)
                    if use_cutmix:
                        inputs, labels_b, lam = _cutmix(inputs, labels, cutmix_alpha, device)
                    else:
                        lam = np.random.beta(mixup_alpha, mixup_alpha)
                        idx = torch.randperm(inputs.size(0), device=device)
                        inputs = lam * inputs + (1 - lam) * inputs[idx]
                        labels_b = labels[idx]

                with torch.set_grad_enabled(is_train):
                    with autocast("cuda", enabled=use_amp):
                        outputs = model(inputs)
                        if is_train and lam < 1.0:
                            loss = lam * criterion(outputs, labels) + (1 - lam) * criterion(outputs, labels_b)
                        else:
                            loss = criterion(outputs, labels)

                    preds = outputs.argmax(dim=1)

                    if is_train:
                        scaler.scale(loss).backward()
                        scaler.step(optimizer)
                        scaler.update()
                        if is_onecycle:
                            scheduler.step()

                running_loss     += loss.item() * inputs.size(0)
                running_corrects += (preds == labels).sum().item()

            epoch_loss = running_loss / dataset_size[phase]
            epoch_acc  = running_corrects / dataset_size[phase]

            print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")
            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(epoch_acc)

            if phase == "val":
                if scheduler is not None and not is_onecycle:
                    if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                        scheduler.step(epoch_loss)
                    else:
                        scheduler.step()

                if epoch_acc > best_acc + min_delta:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1

        print(f"Current LR: {optimizer.param_groups[0]['lr']:.2e}")

        # stop once val accuracy hasn't improved by min_delta for `patience` epochs
        if early_stopping and epochs_no_improve >= patience:
            print(f"\nEarly stopping after {epoch+1} epochs.")
            break

        print()

    elapsed = time.time() - since
    print(f"Training complete in {elapsed // 60:.0f}m {elapsed % 60:.0f}s")
    print(f"Best val Acc: {best_acc:.4f}")

    model.load_state_dict(best_model_wts)  # restore best checkpoint
    return model, history

In [ ]:
def set_trainable(model, train_backbone: bool):
    for p in model.parameters():
        p.requires_grad = train_backbone
    for p in model.classifier.parameters():
        p.requires_grad = True

model_ft = models.convnext_small(weights=models.ConvNeXt_Small_Weights.DEFAULT)
model_ft.classifier[2] = nn.Linear(model_ft.classifier[2].in_features, num_classes)
model_ft = model_ft.to(device)

# Stage 1: freeze the pretrained backbone, train only the new classifier head.
# Avoids large random-init gradients from the new head wrecking pretrained ImageNet features.
set_trainable(model_ft, train_backbone=False)

optimizer_stage1 = optim.AdamW(
    model_ft.classifier.parameters(),
    lr=1e-3,
    weight_decay=0.05,
)

steps_per_epoch = len(dataloaders["train"])
scheduler_stage1 = optim.lr_scheduler.OneCycleLR(
    optimizer_stage1,
    max_lr=1e-3,
    epochs=10,
    steps_per_epoch=steps_per_epoch,
    pct_start=0.3,
)  # OneCycle: warmup+decay tuned for this short, fixed-length phase

model_ft, history1 = train_model(
    model_ft, dataloaders, device,
    criterion=criterion,
    optimizer=optimizer_stage1,
    num_epochs=10,
    scheduler=scheduler_stage1,
    early_stopping=False,
    mixup_alpha=0.2,
    cutmix_alpha=0.2,
)

In [ ]:
# Stage 2: unfreeze the backbone and fine-tune the whole network end-to-end.
set_trainable(model_ft, train_backbone=True)

backbone_params   = [p for n, p in model_ft.named_parameters() if not n.startswith("classifier.")]
classifier_params = list(model_ft.classifier.parameters())

# Differential LR: backbone gets a much smaller LR than the head, so pretrained features
# are only nudged, not overwritten.
optimizer_stage2 = optim.AdamW(
    [
        {"params": classifier_params, "lr": 3e-4},
        {"params": backbone_params,   "lr": 3e-5},
    ],
    weight_decay=0.05,
)

scheduler_stage2 = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_stage2,
    T_max=60,
    eta_min=1e-6,
)  # long cosine decay; early stopping cuts it short once val accuracy plateaus

model_ft, history2 = train_model(
    model_ft, dataloaders, device,
    criterion=criterion,
    optimizer=optimizer_stage2,
    num_epochs=100,
    scheduler=scheduler_stage2,
    early_stopping=True,
    patience=10,
    min_delta=1e-4,
    mixup_alpha=0.2,
    cutmix_alpha=0.2,
)

In [ ]:
torch.save(model_ft.state_dict(), model_save_path)
print("Model saved to", model_save_path)

Model saved to /home/sera/Projects/image_emotion_detection/convnext_small.pth
